# IRIS live backend on a free cloud GPU

Runs the **IRIS backend in live mode** (three 4-bit models resident) on a free
**Kaggle T4 ×2** or **Colab T4** session and exposes it through a **free
cloudflared quick tunnel** — no paid services anywhere. Your laptop then runs
only the frontend:

```bash
cd iris-frontend
IRIS_API_URL=https://<printed-url>.trycloudflare.com npm run dev
```

**Before running:**
- Kaggle: Settings → Accelerator **GPU T4 ×2**, Internet **on**.
- A Hugging Face **read token** from an account that accepted the
  [google/gemma-3-4b-it](https://huggingface.co/google/gemma-3-4b-it) license —
  add it as a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets) or a Colab
  secret of the same name.
- Optional but recommended: attach a finished `decomp_alg2.json` (from the
  decomp_alg2 notebook's output) and set `POOL_JSON` below — the demo then
  answers few-shot from the first question and stored replays work.

Sessions are time-boxed (Kaggle ≈ 12 h); the tunnel URL changes every session.


In [ ]:
# ── config ───────────────────────────────────────────────────────
REPO_URL    = "https://github.com/ktahsinr/RoSE.git"
REPO_REF    = "feature/iris-frontend"
N_QUESTIONS = 250          # first N ChartQAPro factoid questions
POOL_JSON   = None         # e.g. "/kaggle/input/decomp-alg2/decomp_alg2.json"
RESULTS_DIR = None         # folder with decomp_alg2.json/.csv for replay mode

import os, subprocess, sys, time
from pathlib import Path
ON_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
CODE = WORK / "RoSE"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


In [ ]:
# ── HF token (Kaggle or Colab secret named HF_TOKEN) ─────────────
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass
assert token, "add an HF read token as a secret named HF_TOKEN"
os.environ["HF_TOKEN"] = token
print("HF token found")


In [ ]:
# ── code + dependencies (a few minutes) ──────────────────────────
if not CODE.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_REF,
                    REPO_URL, str(CODE)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "-r", str(CODE / "requirements-pinned.txt"),
                "-r", str(CODE / "iris-backend/requirements.txt")], check=True)
print("dependencies installed")


In [ ]:
# ── verify gated access BEFORE any GPU work ──────────────────────
sys.path.insert(0, str(CODE / "chartqapro"))
from decompose.models import check_gated_access
check_gated_access()


In [ ]:
# ── build the first-N factoid dataset (CPU) ──────────────────────
DATA = WORK / "data" / "chartqapro"
if not (DATA / "dataset.json").exists():
    subprocess.run([sys.executable,
                    str(CODE / "scripts/build_chartqapro_dataset.py"),
                    "--n", str(N_QUESTIONS), "--out", str(DATA)], check=True)
print("dataset ready:", DATA / "dataset.json")


In [ ]:
# ── cloudflared (free quick tunnel, no account) ──────────────────
CF = WORK / "cloudflared"
if not CF.exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/"
        "download/cloudflared-linux-amd64", CF)
    CF.chmod(0o755)
print("cloudflared ready")


In [ ]:
# ── launch the backend in LIVE mode (model load ≈ 5–10 min) ──────
env = dict(os.environ,
    IRIS_DEMO_MODE="false",
    IRIS_DATASET_JSON=str(DATA / "dataset.json"),
    IRIS_IMAGE_DIR=str(DATA / "images"),
    IRIS_CORS_ORIGINS="http://localhost:3000,http://127.0.0.1:3000")
if POOL_JSON:   env["IRIS_POOL_PATH"] = POOL_JSON
if RESULTS_DIR: env["IRIS_RESULTS_DIR"] = RESULTS_DIR
backend_log = open(WORK / "backend.log", "w")
backend = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "iris_backend.main:app",
     "--host", "0.0.0.0", "--port", "8000"],
    cwd=CODE / "iris-backend", env=env,
    stdout=backend_log, stderr=subprocess.STDOUT)

import urllib.request, json as _json
print("waiting for the models to load", end="", flush=True)
for _ in range(240):                     # up to 20 min
    try:
        h = _json.load(urllib.request.urlopen(
            "http://localhost:8000/health", timeout=2))
        break
    except Exception:
        print(".", end="", flush=True); time.sleep(5)
else:
    raise RuntimeError("backend never came up — see backend.log")
print()
assert h["mode"] == "live", f"backend fell back to demo mode: {h}"
print("backend is LIVE:", h["models_loaded"], "| pool", h["pool_size"])


In [ ]:
# ── open the tunnel and print the URL ────────────────────────────
import re
tunnel = subprocess.Popen(
    [str(CF), "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break
print("\n" + "═" * 60)
print("IRIS backend URL:", url)
print("═" * 60)
print("\nOn your laptop:\n")
print(f"  cd iris-frontend")
print(f"  IRIS_API_URL={url} npm run dev")
print("\nthen open http://localhost:3000/explorer and press Run live.")


In [ ]:
# ── keep the session alive; shows the backend log as runs happen ──
import itertools
with open(WORK / "backend.log") as f:
    f.seek(0, 2)
    while backend.poll() is None:
        line = f.readline()
        print(line, end="") if line else time.sleep(2)
